# So sánh Model A (bản gốc) và Model B (pipeline mới) trên cùng tập ảnh thực tế

Notebook chạy cả hai model trên **cùng một bộ ảnh có nhãn thật**, sau đó so sánh Accuracy, Macro Precision/Recall/F1, Weighted F1, classification report và confusion matrix.

- Model A: `model/best_model.h5`, từ commit gốc 07/05/2025.
- Model B: `model/best_model.keras`, upload 27/11/2026.
- Mỗi model có thứ tự nhãn riêng. Notebook chuyển dự đoán của Model A về nhãn chuẩn trước khi tính metric.

Tập kiểm thử phải độc lập với dữ liệu huấn luyện và có nhãn được xác nhận. Không có nhãn thật thì không thể kết luận model nào tốt hơn.

In [ ]:
# 1. Kết nối Drive và tải repository
from google.colab import drive, files
drive.mount("/content/drive")

import os, sys, json, zipfile, hashlib, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

REPO_DIR = Path("/content/Plant-Disease-Detection")
REPO_URL = "https://github.com/ShimeKano/Plant-Disease-Detection.git"
if not REPO_DIR.exists():
    !git clone --branch main {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main
    !git -C {REPO_DIR} checkout main
    !git -C {REPO_DIR} reset --hard origin/main

MODEL_DIR = REPO_DIR / "model"
CLASS_NAMES = json.loads((MODEL_DIR / "class_names.json").read_text(encoding="utf-8"))
IMG_SIZE = 128

# Model A: class order generated alphabetically by flow_from_directory.
LEGACY_CLASS_NAMES = sorted([
    "Tomato__Target_Spot", "Tomato_Leaf_Mold",
    "Tomato__Tomato_YellowLeaf__Curl_Virus", "Pepper__bell___healthy",
    "Tomato__Tomato_mosaic_virus", "Tomato_Late_blight",
    "Tomato_Spider_mites_Two_spotted_spider_mite", "Potato___Early_blight",
    "Tomato_Bacterial_spot", "Pepper__bell___Bacterial_spot",
    "Potato___healthy", "Tomato_Early_blight", "Tomato_healthy",
    "Tomato_Septoria_leaf_spot", "Potato___Late_blight"
])
LEGACY_TO_CURRENT = {
    "Tomato_Bacterial_spot": "Tomato__Bacterial_spot",
    "Tomato_Early_blight": "Tomato__Early_blight",
    "Tomato_Late_blight": "Tomato__Late_blight",
    "Tomato_Leaf_Mold": "Tomato__Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato__Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Tomato__Spider_mites_Two_spotted_spider_mite",
    "Tomato_healthy": "Tomato__healthy",
}
def canonical_class(name):
    return LEGACY_TO_CURRENT.get(name, name)

print("TensorFlow:", tf.__version__)
print("Model A:", MODEL_DIR / "best_model.h5")
print("Model B:", MODEL_DIR / "best_model.keras")
print("Canonical classes:", len(CLASS_NAMES))


In [ ]:
# 2. Upload ZIP ảnh thực tế đã gán nhãn
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Hãy upload một file ZIP chứa thư mục ảnh theo từng class.")
ZIP_PATH = Path("/content/field_test.zip")
ZIP_PATH.write_bytes(uploaded[zip_names[0]])
EXTRACT_DIR = Path("/content/field_test_data")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_DIR)

# Find the directory whose immediate subfolders match the model's class names.
candidate_dirs = [EXTRACT_DIR] + [p for p in EXTRACT_DIR.rglob("*") if p.is_dir()]
DATASET_DIR = None
for candidate in candidate_dirs:
    subdirs = {p.name for p in candidate.iterdir() if p.is_dir()}
    if subdirs.intersection(CLASS_NAMES) or subdirs.intersection(LEGACY_CLASS_NAMES):
        DATASET_DIR = candidate
        break
if DATASET_DIR is None:
    raise ValueError("Không tìm thấy thư mục class. Tên folder phải khớp class_names.json.")

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
samples = []
class_counts = {}
unknown_folders = []
for folder in sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()):
    if canonical_class(folder.name) not in CLASS_NAMES:
        unknown_folders.append(folder.name)
        continue
    images = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in extensions]
    canonical_name = canonical_class(folder.name)
    class_counts[canonical_name] = class_counts.get(canonical_name, 0) + len(images)
    samples.extend((p, canonical_name) for p in images)

if not samples:
    raise ValueError("Không tìm thấy ảnh hợp lệ trong các thư mục class.")
print("Dataset root:", DATASET_DIR)
print("Total labeled images:", len(samples))
print("Class counts:", class_counts)
if unknown_folders:
    print("Ignored folders not in class mapping:", unknown_folders)
if len(class_counts) < 2:
    print("WARNING: chỉ có một lớp; kết quả so sánh sẽ không phản ánh đầy đủ khả năng phân biệt lớp.")

In [ ]:
# 3. Phát hiện đúng Model A và Model B
model_paths = [
    MODEL_DIR / "best_model.h5",      # Model A: original 2025
    MODEL_DIR / "best_model.keras",   # Model B: new 2026
]
missing = [p.name for p in model_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Thiếu model: {missing}")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for p in model_paths:
    print(p.name, "SHA256:", sha256_file(p))


In [ ]:
# 4. Chạy inference cho cả hai model trên cùng ảnh
X = []
y_true_names = []
for path, label in samples:
    try:
        image = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        X.append(np.asarray(image, dtype=np.float32) / 255.0)
        y_true_names.append(label)
    except Exception as exc:
        print("Skip unreadable image:", path, exc)

X = np.asarray(X, dtype=np.float32)
y_true = np.asarray([CLASS_NAMES.index(name) for name in y_true_names], dtype=np.int32)
if len(X) == 0:
    raise RuntimeError("Không có ảnh nào đọc được.")

results = {}
for model_path in model_paths:
    print("\nEvaluating:", model_path.name)
    model = tf.keras.models.load_model(model_path, compile=False)
    source_classes = LEGACY_CLASS_NAMES if model_path.name == "best_model.h5" else CLASS_NAMES
    if int(model.output_shape[-1]) != len(source_classes):
        raise ValueError(f"{model_path.name}: {model.output_shape[-1]} outputs nhưng mapping có {len(source_classes)} classes.")
    probs = model.predict(X, batch_size=32, verbose=1)
    raw_pred = np.argmax(probs, axis=1)
    mapped_names = [canonical_class(source_classes[int(i)]) for i in raw_pred]
    unknown = sorted(set(mapped_names) - set(CLASS_NAMES))
    if unknown:
        raise ValueError(f"{model_path.name}: nhãn không ánh xạ được: {unknown}")
    y_pred = np.asarray([CLASS_NAMES.index(name) for name in mapped_names], dtype=np.int32)
    results[model_path.name] = {"probabilities": probs, "predictions": y_pred}
    del model

if len(results) != 2:
    raise RuntimeError("Không đánh giá được đủ Model A và Model B.")


In [ ]:
# 5. Bảng metric tổng hợp và độ đồng thuận
summary_rows = []
for name, result in results.items():
    pred = result["predictions"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, labels=np.unique(y_true), average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_true, pred, average="weighted", zero_division=0
    )
    summary_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "weighted_f1": weighted_f1,
    })

summary = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
display(summary.style.format({c:"{:.4f}" for c in summary.columns if c != "model"}))

names = list(results)
if len(names) == 2:
    agreement = np.mean(results[names[0]]["predictions"] == results[names[1]]["predictions"])
    print(f"Prediction agreement: {agreement*100:.2f}%")
print("Metric cao hơn trên tập này không đồng nghĩa model tốt hơn trên mọi địa bàn/cây trồng.")

In [ ]:
# 6. Báo cáo chi tiết từng model và lưu artifacts
REPORT_DIR = Path("/content/model_comparison_report")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for name, result in results.items():
    safe = Path(name).stem.replace(" ", "_")
    report = classification_report(
        y_true, result["predictions"],
        labels=np.unique(y_true),
        target_names=[CLASS_NAMES[i] for i in np.unique(y_true)],
        zero_division=0,
        output_dict=True
    )
    (REPORT_DIR / f"{safe}_classification_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    pd.DataFrame(
        confusion_matrix(y_true, result["predictions"], labels=np.unique(y_true)),
        index=[CLASS_NAMES[i] for i in np.unique(y_true)],
        columns=[CLASS_NAMES[i] for i in np.unique(y_true)]
    ).to_csv(REPORT_DIR / f"{safe}_confusion_matrix.csv", encoding="utf-8-sig")

summary.to_csv(REPORT_DIR / "model_comparison_summary.csv", index=False, encoding="utf-8-sig")
run_info = {
    "dataset_root_name": DATASET_DIR.name,
    "image_count": int(len(y_true)),
    "class_counts": class_counts,
    "models": list(results.keys()),
    "metrics": summary.to_dict(orient="records"),
    "note": "Evaluation on a user-supplied labeled test set. Verify labels and ensure no overlap with training data."
}
(REPORT_DIR / "evaluation_metadata.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8"
)
archive = shutil.make_archive("/content/model_comparison_report", "zip", REPORT_DIR)
print("Report ZIP:", archive)
files.download(archive)

## Diễn giải kết quả

- Ưu tiên Macro F1 khi số ảnh mỗi lớp mất cân bằng; xem thêm recall từng lớp nếu bỏ sót bệnh là rủi ro lớn.
- Accuracy có thể bị chi phối bởi lớp có nhiều ảnh.
- Nếu hai model dự đoán giống nhau, điều đó không chứng minh chúng độc lập hoặc chính xác.
- Nếu dataset nhỏ, chỉ có vài lớp, hoặc nhãn không được chuyên gia xác nhận, kết quả chỉ mang tính thăm dò.
- Nên báo cáo riêng kết quả PlantVillage và kết quả ảnh thực địa. Không gộp hai bộ dữ liệu thành một con số duy nhất nếu chưa thiết kế protocol phù hợp.

**Mapping nhãn:** Model A dùng thứ tự lớp alphabetic của thư mục PlantVillage trong notebook cũ; Model B dùng `model/class_names.json`. Notebook ánh xạ tên lớp cũ sang nhãn chuẩn trước khi tính metric.